# Module 11 Lab - Hyperparameter Tuning & AutoML**Objective:** To learn how to optimize model performance by tuning **hyperparameters** and to get an introduction to the powerful concept of **Automated Machine Learning (AutoML)**.**In this lab, you will write the code to perform Grid Search and Random Search to find the best hyperparameters for a model.**

## Part 1: What are Hyperparameters?**Concept:** In machine learning, there are two types of parameters:1.  **Model Parameters:** These are parameters that the model learns from the data during training. For example, the coefficients in a Linear Regression model.2.  **Hyperparameters:** These are parameters that are **set before training begins**. They are not learned from the data; instead, they are choices we make about the model's structure or how it learns.     *   *Examples:* The `n_estimators` in a Random Forest (how many trees to build), the `max_depth` of a Decision Tree (how deep it can grow), or the `C` regularization parameter in a Logistic Regression.Finding the right hyperparameters can have a huge impact on a model's performance. **Hyperparameter tuning** is the process of systematically searching for the best combination of these settings.

## Part 2: SetupWe will use the Iris dataset and a `RandomForestClassifier`, which has several important hyperparameters we can tune.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
# Load and prepare data
iris = load_iris()
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# A baseline model with default hyperparameters
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)
print(f"Accuracy of baseline Random Forest: {accuracy_baseline:.2%}")

Accuracy of baseline Random Forest: 100.00%


## Part 3: Grid Search**Concept:** Grid Search is the most straightforward tuning method. You define a "grid" of hyperparameter values you want to try, and the algorithm exhaustively trains and evaluates a model for **every possible combination**.*   **Pro:** It's guaranteed to find the best combination within the grid.*   **Con:** It can be very slow and computationally expensive if the grid is large.

### Task 1: Perform a Grid Search**Your Task:** Use `GridSearchCV` from `sklearn.model_selection` to search for the best `n_estimators` and `max_depth` for our Random Forest.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
# Load and prepare data
iris = load_iris()
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 1. Define the grid of hyperparameters to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None]
}

# 2. Create a GridSearchCV instance
# n_jobs=-1 uses all available processors to speed up the search
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

# 3. Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# 4. Print the best parameters and the best score
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best cross-validated score: {grid_search.best_score_:.2%}")

# 5. Access the best model directly for final predictions
best_rf_model = grid_search.best_estimator_

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Parameters: {'max_depth': 5, 'n_estimators': 100}
Best cross-validated score: 94.29%


## Part 4: Random Search**Concept:** Random Search is often more efficient than Grid Search. Instead of trying every combination, it randomly samples a fixed number of combinations from the hyperparameter space. *   **Pro:** It's much faster and can explore a wider range of values.*   **Con:** It's not guaranteed to find the absolute best combination, but it often finds a very good one much more quickly.

### Task 2: Perform a Random Search**Your Task:** Use `RandomizedSearchCV` to perform a random search over a larger hyperparameter space.

In [8]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the distribution of hyperparameters
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=50, stop=500, num=10)],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

# 2. Create a RandomizedSearchCV instance
# Using n_iter=10 to randomly sample 10 combinations over 5-fold CV
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# 3. Fit the random search to the data
random_search.fit(X_train, y_train)

# 4. Print the results
print(f"Best Parameters: {random_search.best_params_}")
print(f"Best cross-validated score: {random_search.best_score_:.2%}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best cross-validated score: 94.29%


## Part 5: Introduction to AutoML with AutoGluon**Concept:** AutoML takes hyperparameter tuning to the next level. It automates the entire ML workflow, including:*   Data preprocessing*   Feature engineering*   Model selection (trying many different types of models)*   Hyperparameter tuning*   Ensemble creation**AutoGluon** is a popular and easy-to-use AutoML library. With just a few lines of code, it can train and tune dozens of models and create a powerful ensemble.**This part is fully coded.** Your task is to run it and see the power of AutoML. Note that it may take a few minutes to run.

In [11]:
# You may need to install AutoGluon first. Uncomment the line below in your Colab notebook.
!pip install autogluon
from autogluon.tabular import TabularPredictor
# AutoGluon requires the data in a single DataFrame with the target column.
# We will create a training DataFrame for AutoGluon
train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag['species'] = y_train
# Create a test DataFrame as well
test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag['species'] = y_test
# Create and train the TabularPredictor
# `time_limit=60` tells it to run for 60 seconds
# Create and train the TabularPredictor
predictor = TabularPredictor(label='species', eval_metric='accuracy')
predictor.fit(train_data=train_data_ag, time_limit=60)

# Evaluate the predictor on the test data
leaderboard = predictor.leaderboard(test_data_ag)
print(leaderboard)

No path specified. Models will be saved in: "AutogluonModels/ag-20260402_175326"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.77 GB / 12.67 GB (85.0%)
Disk Space Avail:   74.74 GB / 107.72 GB (69.4%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.

                  model  score_test  score_val eval_metric  pred_time_test  \
0               XGBoost    1.000000   0.952381    accuracy        0.033596   
1        ExtraTreesEntr    1.000000   0.904762    accuracy        0.093961   
2      RandomForestEntr    1.000000   0.857143    accuracy        0.100800   
3      RandomForestGini    1.000000   0.857143    accuracy        0.100965   
4        ExtraTreesGini    1.000000   0.904762    accuracy        0.110112   
5         LightGBMLarge    0.977778   0.857143    accuracy        0.001826   
6              LightGBM    0.955556   0.857143    accuracy        0.001209   
7        NeuralNetTorch    0.955556   0.952381    accuracy        0.011697   
8            LightGBMXT    0.933333   0.952381    accuracy        0.001709   
9   WeightedEnsemble_L2    0.933333   0.952381    accuracy        0.004502   
10             CatBoost    0.933333   0.952381    accuracy        0.004955   
11      NeuralNetFastAI    0.866667   0.904762    accuracy      

## 📝 Knowledge Check**Instructions:** Answer the following questions in this markdown cell.
1.  **What is the main difference between a model parameter and a hyperparameter?**
2.  **When would you choose to use Grid Search over Random Search, and vice-versa?**
3.  **Looking at the AutoGluon leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?**

**[ENTER YOUR ANSWERS HERE]**
1. Model parameter is are parameters that the model learns from the data during training, while hyperparameters are parameters that are set before training begins

2. Grid search can guaranteed to find the best combination within the grid, but it can be very slow and computationally expensive if the grid is large. Random search is much faster and can explore a wider range of values, but while it can't guaranteed to find the absolute best combination, it often finds a very good one much more quickly

3. The model that preformed the best is WeightedEnsemble_L2. When using the best_quality preset, AutoGluon combines multiple base models into a weighted ensemble, which often outperforming any single tuned model.